In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
import fresnel
import ipywidgets as widgets

from IPython.display import Image, display
from md_Helpers.lattices import (
    build_fcc_lattice,
    make_gsd_frame,
)


# ============================================================
# Lattice settings
# ============================================================

n_fcc_cells = 2
target_rho = 0.8
particle_type = "A"


# ============================================================
# Build the lattice using the V4 API
# ============================================================

lattice = build_fcc_lattice(
    n_cells=n_fcc_cells,
    density=target_rho,
)

frame = make_gsd_frame(
    lattice,
    particle_type=particle_type,
)

positions = np.asarray(
    frame.particles.position,
    dtype=float,
)

box = np.asarray(
    frame.configuration.box,
    dtype=float,
)

box_lengths = box[:3]
L = max(box_lengths)


# ============================================================
# Determine which FCC unit cell contains each particle
#
# This is the same cell-assignment method used in V3.
# ============================================================

cell_size = box_lengths / n_fcc_cells

cell_indices = np.floor(
    (positions + box_lengths / 2) / cell_size
).astype(int)

# Protect against floating-point boundary errors
cell_indices = np.clip(
    cell_indices,
    0,
    n_fcc_cells - 1,
)

# Convert each (x, y, z) cell index into one unique integer
cell_ids = np.ravel_multi_index(
    cell_indices.T,
    dims=(
        n_fcc_cells,
        n_fcc_cells,
        n_fcc_cells,
    ),
)

number_of_cells = n_fcc_cells**3


# ============================================================
# Assign a different color to every FCC unit cell
# ============================================================

color_map = plt.colormaps["turbo"]

unit_cell_colors = color_map(
    np.linspace(0, 1, number_of_cells)
)[:, :3]

particle_colors = unit_cell_colors[cell_ids]


# Verify that the 2x2x2 lattice contains eight cells
# with four FCC particles assigned to each cell.
unique_cell_ids, particles_per_cell = np.unique(
    cell_ids,
    return_counts=True,
)

if len(unique_cell_ids) != number_of_cells:
    raise RuntimeError(
        f"Found {len(unique_cell_ids)} cells; "
        f"expected {number_of_cells}."
    )

if not np.all(particles_per_cell == 4):
    raise RuntimeError(
        "Each conventional FCC cell should contain four particles. "
        f"Observed counts: {particles_per_cell.tolist()}"
    )

print(f"FCC cells per side: {n_fcc_cells}")
print(f"Total FCC cells: {number_of_cells}")
print(f"Particles per cell: {particles_per_cell.tolist()}")
print(f"Total particles: {lattice.n_particles}")
print(f"Target density: {lattice.target_density}")
print(f"Actual density: {lattice.actual_density}")
print(f"Box length: {lattice.box_length}")


# ============================================================
# Fresnel setup
# ============================================================

device = fresnel.Device()

tracer = fresnel.tracer.Path(
    device=device,
    w=500,
    h=500,
)


def render_lattice(azimuth, elevation, view_scale):
    scene = fresnel.Scene(device)

    particles = fresnel.geometry.Sphere(
        scene,
        N=len(positions),
        radius=0.5,
    )

    particles.position[:] = positions

    particles.material = fresnel.material.Material(
        roughness=0.5,
        primitive_color_mix=1.0,
    )

    particles.color[:] = fresnel.color.linear(
        particle_colors
    )

    particles.outline_width = 0.04

    fresnel.geometry.Box(
        scene,
        box,
        box_radius=0.025,
    )

    azimuth_radians = math.radians(azimuth)
    elevation_radians = math.radians(elevation)

    camera_distance = 2.5 * L

    camera_position = camera_distance * np.array([
        math.cos(elevation_radians)
        * math.cos(azimuth_radians),

        math.cos(elevation_radians)
        * math.sin(azimuth_radians),

        math.sin(elevation_radians),
    ])

    scene.camera = fresnel.camera.Orthographic(
        position=tuple(camera_position),
        look_at=(0, 0, 0),
        up=(0, 0, 1),
        height=L * view_scale,
    )

    scene.lights = fresnel.light.lightbox()
    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1

    image = tracer.sample(
        scene,
        samples=256,
    )

    display(Image(image._repr_png_()))


# ============================================================
# Interactive camera controls
# ============================================================

azimuth_slider = widgets.FloatSlider(
    value=-70,
    min=-180,
    max=180,
    step=5,
    description="Azimuth",
    continuous_update=False,
)

elevation_slider = widgets.FloatSlider(
    value=15,
    min=-80,
    max=80,
    step=5,
    description="Elevation",
    continuous_update=False,
)

view_scale_slider = widgets.FloatSlider(
    value=1.6,
    min=0.7,
    max=2.5,
    step=0.1,
    description="Zoom out",
    continuous_update=False,
)

output = widgets.interactive_output(
    render_lattice,
    {
        "azimuth": azimuth_slider,
        "elevation": elevation_slider,
        "view_scale": view_scale_slider,
    },
)

display(
    widgets.VBox([
        azimuth_slider,
        elevation_slider,
        view_scale_slider,
        output,
    ])
)

FCC cells per side: 2
Total FCC cells: 8
Particles per cell: [4, 4, 4, 4, 4, 4, 4, 4]
Total particles: 32
Target density: 0.8
Actual density: 0.8000000000000003
Box length: 3.4199518933533937
